In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

account_data = [
    ["3050XXX982", "NAKHON NAYOK", "Savings"],
    ["9597XXX468", "NAKHON NAYOK", "Savings"],
    ["1022733936", "ดินแดง", "Savings"],
    ["3164155931", "THE FOURTH", "Savings"],
    ["6664038417", "โคราช", "Savings"],
    ["4142174890", "ขอนแก่น", "Savings"],
    ["4282216316", "คลอง 3", "Savings"],
    ["0172762156", "ท่าพระ", "Savings"],
    ["5194245562", "นครปฐม", "Savings"],
    ["6402700634", "บางแสน", "Savings"],
    ["4271897496", "บางใหญ่", "Savings"],
    ["4221594268", "บางนา", "Savings"],
    ["1922438369", "บางพลัด", "Savings"],
    ["1564296424", "พระราม 2", "Savings"],
    ["6692694546", "พัทยา", "Savings"],
    ["3702819949", "รังสิต", "Savings"],
    ["1362714171", "รามคำแหง", "Savings"],
    ["4122310173", "วังหิน", "Savings"],
    ["6041259598", "NAKHON NAYOK", "Savings"],
    ["168-2-93045-0", "NAKHON NAYOK", "Current Account"],
    ["053-1-97925-7", "NAKHON NAYOK", "Savings"],
    ["075-8-56737-0", "รามอินทรา", "Savings"],
    ["132-1-07885-7", "NAKHON NAYOK", "Credit Card"],
]

dim_account = pd.DataFrame(
    account_data,
    columns=[
        "account_number",
        "branch",
        "account_type"
    ]
)

# Clean text
for column in ["account_number", "branch", "account_type"]:
    dim_account[column] = (
        dim_account[column]
        .astype("string")
        .str.strip()
    )

# Standardize English branch names
dim_account["branch"] = dim_account["branch"].replace({
    "the fourth": "THE FOURTH",
    "The Fourth": "THE FOURTH"
})

# Ensure one row per account
dim_account = (
    dim_account
    .drop_duplicates(subset=["account_number"])
    .sort_values("account_number")
    .reset_index(drop=True)
)

# Create surrogate key
dim_account.insert(
    0,
    "account_key",
    range(1, len(dim_account) + 1)
)

display(dim_account)
print(dim_account.dtypes)

c:\Users\User\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


,account_key,account_number,branch,account_type
0,1,0172762156,ท่าพระ,Savings
1,2,053-1-97925-7,NAKHON NAYOK,Savings
2,3,075-8-56737-0,รามอินทรา,Savings
3,4,1022733936,ดินแดง,Savings
4,5,132-1-07885-7,NAKHON NAYOK,Credit Card
5,6,1362714171,รามคำแหง,Savings
6,7,1564296424,พระราม 2,Savings
7,8,168-2-93045-0,NAKHON NAYOK,Current Account
8,9,1922438369,บางพลัด,Savings
9,10,3050XXX982,NAKHON NAYOK,Savings


account_key        int64
account_number    string
branch            string
account_type      string
dtype: object


In [2]:
current_folder = Path.cwd()

if current_folder.name.lower() == "python_command":
    project_root = current_folder.parent
else:
    project_root = current_folder

dim_folder = project_root / "Datamart" / "Dim"
dim_folder.mkdir(parents=True, exist_ok=True)

database_path = dim_folder / "dim_database.db"

with sqlite3.connect(database_path) as connection:
    dim_account.to_sql(
        name="dim_account",
        con=connection,
        if_exists="replace",
        index=False
    )

print(f"Database: {database_path.resolve()}")
print("Table: dim_account")
print(f"Rows exported: {len(dim_account):,}")

Database: C:\Users\User\Desktop\Python 100 Days\Bank_Account_Project\Datamart\Dim\dim_database.db
Table: dim_account
Rows exported: 23
